# EEU4C16/EEP5C16 Lab 7 - Fast Single Image Super-Resolution

## Background
Super-resolution is a technique in image and video processing whereby a low resolution (LR) image/video frame is upsampled to a higher resolution (HR). This has numerous applications for:

- Balancing image/video quality and transmission efficiency for websites.

- Upscaling of photos taken on mobile devices ([Google Research Blog - Enhance! RAISR Sharp Images with Machine Learning](https://research.google/blog/enhance-raisr-sharp-images-with-machine-learning/)).

- Upscaling of video games ([NVIDIA - DLSS 4](https://www.nvidia.com/en-us/geforce/technologies/dlss/)).

Fast super-resolution is of utmost importance for real-time applications and a number of companies are looking into this: [Qualcomm - QuickSRNet: Plain Single-Image Super-Resolution Architecture for Faster Inference on Mobile Platforms](https://openaccess.thecvf.com/content/CVPR2023W/MobileAI/papers/Berger_QuickSRNet_Plain_Single-Image_Super-Resolution_Architecture_for_Faster_Inference_on_Mobile_CVPRW_2023_paper.pdf)

## Your Task
Train a super-resolution model to up-sample image patches from $32\times32$ to $128\times128$ (a factor of $\times4$) and get as close to real-time processing as possible (take this to mean processing a $30$ $fps$ video stream which means your model inference time should be close to $1/30$ seconds $~0.03$ $s$).

**Data Pre-Processing Requirements:**

1. You will be given a training dataset that is represenatitve of the target upsampling distribution (you may want to augment this with other datasets you can find online).

2. You will have to downsample these images and use them as input to the super-resolution neural network.

3. Generate an approriate training/test split from your dataset.

**Architecture Requirements:**

1. Design and train a deep neural network architecture that finds a tradeoff between speed and accuracy.

2. You will need to implement the model class (`SuperResolutionModel`) in (`models.py`) and save the model weights as `model.pth` (**these class and filenames must be left unchanged**).

Finally you will create a report on your data processing/network design process. Try and create some nice informative graphs, images and tables for this.

## Important Notes (Read Carefully!)
- The evaluation on the server will record measures of [PSNR](https://en.wikipedia.org/wiki/Peak_signal-to-noise_ratio), [SSIM](https://en.wikipedia.org/wiki/Structural_similarity_index_measure), as well as parameter count (should be **below 5 million params**). You should talk about efficiency runtime etc. in your report. Note that you are free to consider other image quality metrics. You can do some research about these other metrics and consider them for discussion in your report. You can also consider loss functions that go beyond the simple $L_1$/$L_2$ losses.

- Note that the human visual system is more sensitive to changes in luminance
  than chrominance ([Wikipedia - Luma
  (video)](https://en.wikipedia.org/wiki/Luma_(video))). You can take advantage
  of this.

- If you choose to use you own dataset you must save it to a directory on the
  Colab VM as we have done in the cell below i.e.
  `/home/tcd/super_resolution/dataset.npz`, **DO NOT** save it anywhere in the
  `4c16-labs/code` directory as it will affect the whole git system.

- If you are making your own dataset, please chose seperate temporory working
  directory outside `4c16-labs/code`. You could make as `4c16-labs/dataset` to
  store. You CANNOT train and use dataset directly from Google Drive, you SHOULD
  move to Google Colab VM dataset (eg. `home/tcd/super_resolution` ) every session.

- There are different conventions for specifying the dimensions of input/output tensors i.e. $(B, H, W, C)$ or $(B, C, H, W)$ where:    
    - $B=$ batch index
    - $H=$ image height
    - $W=$ image width
    - $C=$ image channels (generally $3$ e.g. $RGB$, $YUV$)

- We expect your network to have $(B, C, H, W)$ (channel first convention used by torch layers) as the input and output tensor dimensions

- Your score for this lab is not directly tied to your performance on the backend. It will be a combination of your network performance and how you present your design in the interview.

In [1]:
# This cell mounts Colab to your Google Drive and navigates to the script directory
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive/4c16-labs/code/lab-07/

Mounted at /content/gdrive
/content/gdrive/MyDrive/4c16-labs/code/lab-07


In [2]:
# Create a .gitignore file to avoid committing large files to git accidentlly
# You should run this cell only once at start of the lab.
# c.f. https://git-scm.com/docs/gitignore
%%writefile .gitignore
*.pyc
__pycache__/
.png
.jpg
.zip
.tar
.tar*

Overwriting .gitignore


In [3]:
# Download the dataset (should take about ~2 minutes, 617MB)
!curl --create-dirs -o /home/tcd/super_resolution/dataset.npz https://tcddeeplearning.blob.core.windows.net/deeplearning202324/hr_images.npz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  617M  100  617M    0     0  6727k      0  0:01:33  0:01:33 --:--:-- 7676k


In [8]:
# Import necessary functions libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Import our model definition (if you get a module import error make sure your
# current working directory is the script directory is 4c16-labs/code/lab-07/
# from your Google Drive
from models import SuperResolutionModel

# Dataset pre-processing
Here we load the dataset we have provided from a Numpy compressed (`.npz`) file. It's up to you to pre-process the dataset.

In [10]:
import numpy as np
import torch
from skimage.transform import rescale
from torch.utils.data import Dataset, DataLoader, random_split

# Load the high-res images
hr_images = np.load('/home/tcd/super_resolution/dataset.npz')['hr_images']
print(f"Loaded HR images shape: {hr_images.shape}")

# Normalize to [0, 1]
hr_images = hr_images / 255.0
hr_images = hr_images.astype(np.float32)

# Downsample to create LR images (128x128 → 32x32)
print("Creating low-resolution images...")
lr_images = []
for i in range(len(hr_images)):
    # Downsample by factor of 4 (128 → 32)
    lr_img = rescale(hr_images[i], 0.25, anti_aliasing=True, channel_axis=2)
    lr_images.append(lr_img)
    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1}/{len(hr_images)} images")

lr_images = np.array(lr_images, dtype=np.float32)
print(f"LR images shape: {lr_images.shape}")

# Convert to PyTorch format (B, C, H, W)
hr_images_torch = torch.from_numpy(hr_images).permute(0, 3, 1, 2)
lr_images_torch = torch.from_numpy(lr_images).permute(0, 3, 1, 2)

print(f"HR tensor shape: {hr_images_torch.shape}")
print(f"LR tensor shape: {lr_images_torch.shape}")

# Save processed data
torch.save({
    'hr_images': hr_images_torch,
    'lr_images': lr_images_torch
}, '/home/tcd/super_resolution/hr_lr_pairs.pt')
print("Saved processed dataset!")

Loaded HR images shape: (17354, 128, 128, 3)
Creating low-resolution images...
Processed 1000/17354 images
Processed 2000/17354 images
Processed 3000/17354 images
Processed 4000/17354 images
Processed 5000/17354 images
Processed 6000/17354 images
Processed 7000/17354 images
Processed 8000/17354 images
Processed 9000/17354 images
Processed 10000/17354 images
Processed 11000/17354 images
Processed 12000/17354 images
Processed 13000/17354 images
Processed 14000/17354 images
Processed 15000/17354 images
Processed 16000/17354 images
Processed 17000/17354 images
LR images shape: (17354, 32, 32, 3)
HR tensor shape: torch.Size([17354, 3, 128, 128])
LR tensor shape: torch.Size([17354, 3, 32, 32])
Saved processed dataset!


# Defining the Dataset Generator and Dataloader
Next we define the dataloader which will give us an iterable object of high-resolution and low-resolution image pairs to use in our training loop. You can define data augmentation strategies to randomly flip image pairs, shift pixel values etc.

In [11]:
class SuperResolutionDataset(Dataset):
    def __init__(self, lr_images, hr_images):
        self.lr_images = lr_images
        self.hr_images = hr_images

    def __len__(self):
        return len(self.hr_images)

    def __getitem__(self, idx):
        return self.lr_images[idx], self.hr_images[idx]

# Create dataset
full_dataset = SuperResolutionDataset(lr_images_torch, hr_images_torch)

# Split into train and validation (90/10 split)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Train size: {train_size}, Val size: {val_size}")

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

Train size: 15618, Val size: 1736


# Next steps
*   Set up your DataLoader etc. and think about augmentations, if you want to gather more datasets for training etc.
*   Set up your model architecture and training loop to see how your initial design performs and iterate on this to increase performance.
*   You should think about how you are going to plot results and show network performance in your report.

In [ ]:
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Initialize model
from models import SuperResolutionModel
import importlib
import models
importlib.reload(models)
from models import SuperResolutionModel

sr_model = SuperResolutionModel()

# Check parameter count
total_params = sum(p.numel() for p in sr_model.parameters())
print(f"Total parameters: {total_params:,}")
if total_params > 5_000_000:
    raise Exception("Model has too many parameters!")

# Setup training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
sr_model = sr_model.to(device)

criterion = nn.L1Loss()  # L1 loss often works better than MSE for images
optimizer = optim.Adam(sr_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# Training loop
num_epochs = 50
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    sr_model.train()
    train_loss = 0

    for lr_batch, hr_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]', leave=False):
        lr_batch = lr_batch.to(device)
        hr_batch = hr_batch.to(device)

        optimizer.zero_grad()
        sr_batch = sr_model(lr_batch)
        loss = criterion(sr_batch, hr_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * lr_batch.size(0)

    train_loss = train_loss / len(train_dataset)
    train_losses.append(train_loss)

    # Validation phase
    sr_model.eval()
    val_loss = 0

    with torch.no_grad():
        for lr_batch, hr_batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]', leave=False):
            lr_batch = lr_batch.to(device)
            hr_batch = hr_batch.to(device)

            sr_batch = sr_model(lr_batch)
            loss = criterion(sr_batch, hr_batch)
            val_loss += loss.item() * lr_batch.size(0)

    val_loss = val_loss / len(val_dataset)
    val_losses.append(val_loss)

    scheduler.step(val_loss)

    print(f'Epoch {epoch+1}: Train Loss = {train_loss:.6f}, Val Loss = {val_loss:.6f}')

    # Plot every 5 epochs
    if (epoch + 1) % 5 == 0:
        plt.figure(figsize=(10, 4))
        plt.plot(train_losses, label='Train Loss')
        plt.plot(val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.title('Training Progress')
        plt.show()

print("Training complete!")

Total parameters: 779,011
Using device: cpu


Validation

In [ ]:
# Save the complete model
torch.save(sr_model, 'model.pth')
print("Model saved as model.pth")

# Verify parameter count one more time
total_params = sum(p.numel() for p in sr_model.parameters())
print(f"Final parameter count: {total_params:,}")

# Load model and test
loaded_model = torch.load('model.pth', map_location=device)
loaded_model.eval()

# Get a few test samples
test_lr, test_hr = next(iter(val_loader))
test_lr = test_lr[:4].to(device)
test_hr = test_hr[:4].to(device)

with torch.no_grad():
    test_sr = loaded_model(test_lr)

# Visualize results
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    # Low-res input (upsampled for visualization)
    lr_img = F.interpolate(test_lr[i:i+1], size=(128, 128), mode='bilinear')[0]
    lr_img = lr_img.cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(np.clip(lr_img, 0, 1))
    axes[0, i].set_title('LR (Bicubic)')
    axes[0, i].axis('off')

    # Super-resolved
    sr_img = test_sr[i].cpu().permute(1, 2, 0).numpy()
    axes[1, i].imshow(np.clip(sr_img, 0, 1))
    axes[1, i].set_title('Super-Resolved')
    axes[1, i].axis('off')

    # Ground truth
    hr_img = test_hr[i].cpu().permute(1, 2, 0).numpy()
    axes[2, i].imshow(np.clip(hr_img, 0, 1))
    axes[2, i].set_title('Ground Truth')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

# Measure inference time
times = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        _ = loaded_model(test_lr[:1])
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    times.append(time.time() - start)

avg_time = np.mean(times)
fps = 1 / avg_time
print(f"Average inference time: {avg_time*1000:.2f}ms")
print(f"FPS: {fps:.2f}")
print(f"Real-time target (30fps = 33.3ms): {'✓ PASS' if avg_time < 0.0333 else '✗ FAIL'}")

# IMPORTANT
Make sure your model definition in `models.py` has the class name
`SuperResolutionModel` and make sure you save your weights as `model.pth` e.g.
`torch.save(model, 'model.pth')`.

**DO NOT** submit a model which has more than 5 Million parameters. We will
reject in the backend.

In [ ]:
def count_model_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_model_parameters(sr_model)
if total_params > 5000000:
    print("Model greater than 5 million params!! Reduce model complexity!!")
else:
    torch.save(sr_model, 'model.pth')